<a href="https://colab.research.google.com/github/NehalShahu/Gen_AI/blob/main/Gen_AI_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers sentencepiece sacrebleu pandas torch

In [2]:
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import sacrebleu

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [3]:
FILE_PATH = "/content/hindi_english_parallel.csv"

df = pd.read_csv(FILE_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (1561841, 2)

Columns:
['hindi', 'english']

First 5 rows:


,hindi,english
0,अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें,Give your application an accessibility workout
1,एक्सेर्साइसर पहुंचनीयता अन्वेषक,Accerciser Accessibility Explorer
2,निचले पटल के लिए डिफोल्ट प्लग-इन खाका,The default plugin layout for the bottom panel
3,ऊपरी पटल के लिए डिफोल्ट प्लग-इन खाका,The default plugin layout for the top panel
4,उन प्लग-इनों की सूची जिन्हें डिफोल्ट रूप से नि...,A list of plugins that are disabled by default


In [4]:
ENGLISH_COLUMN = "english"
HINDI_COLUMN = "hindi"

data = df[[ENGLISH_COLUMN, HINDI_COLUMN]].copy()

data.columns = ["english", "hindi"]

# Remove missing values
data = data.dropna()

# Convert to string
data["english"] = data["english"].astype(str)
data["hindi"] = data["hindi"].astype(str)

# Remove extra spaces
data["english"] = data["english"].str.strip()
data["hindi"] = data["hindi"].str.strip()

# Remove empty rows
data = data[
    (data["english"] != "") &
    (data["hindi"] != "")
]

# Remove duplicates
data = data.drop_duplicates()

print("Clean dataset size:", len(data))

display(data.head())

Clean dataset size: 1331302


,english,hindi
0,Give your application an accessibility workout,अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें
1,Accerciser Accessibility Explorer,एक्सेर्साइसर पहुंचनीयता अन्वेषक
2,The default plugin layout for the bottom panel,निचले पटल के लिए डिफोल्ट प्लग-इन खाका
3,The default plugin layout for the top panel,ऊपरी पटल के लिए डिफोल्ट प्लग-इन खाका
4,A list of plugins that are disabled by default,उन प्लग-इनों की सूची जिन्हें डिफोल्ट रूप से नि...


In [5]:
MODEL_NAME = "Helsinki-NLP/opus-mt-en-hi"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Loading translation model...")

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

print("Model loaded successfully!")

Loading tokenizer...


config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  306MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Model loaded successfully!


In [6]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

model = model.to(device)

Using device: cuda


In [7]:
def translate_text(text):

    # Tokenize English input
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )

    # Move input to GPU/CPU
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Generate Hindi translation
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

    # Convert tokens back to text
    translation = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    return translation

In [8]:
test_sentences = [
    "How are you?",
    "What is your name?",
    "I am going to school.",
    "I love India.",
    "Good morning.",
    "Thank you.",
    "Where are you going?"
]

for sentence in test_sentences:

    translation = translate_text(sentence)

    print("English :", sentence)
    print("Hindi   :", translation)
    print("-" * 60)

English : How are you?
Hindi   : आप कैसे हैं?
------------------------------------------------------------
English : What is your name?
Hindi   : आपका Windows Live कूटशब्द क्या है?
------------------------------------------------------------
English : I am going to school.
Hindi   : मैं स्कूल जा रहा हूँ.
------------------------------------------------------------
English : I love India.
Hindi   : मैं भारत से प्यार करता हूँ.
------------------------------------------------------------
English : Good morning.
Hindi   : सुप्रभात.
------------------------------------------------------------
English : Thank you.
Hindi   : धन्यवाद.
------------------------------------------------------------
English : Where are you going?
Hindi   : तुम कहाँ जा रहे हो?
------------------------------------------------------------


In [9]:
from sacrebleu.metrics import BLEU

bleu = BLEU()

sample_size = min(100, len(data))

evaluation_data = data.sample(
    sample_size,
    random_state=42
)

predictions = []
references = []

for i, row in evaluation_data.iterrows():

    english = row["english"]
    actual_hindi = row["hindi"]

    predicted_hindi = translate_text(english)

    predictions.append(predicted_hindi)
    references.append(actual_hindi)

    print(f"Processed {len(predictions)}/{sample_size}")

Processed 1/100
Processed 2/100
Processed 3/100
Processed 4/100
Processed 5/100
Processed 6/100
Processed 7/100
Processed 8/100
Processed 9/100
Processed 10/100
Processed 11/100
Processed 12/100
Processed 13/100
Processed 14/100
Processed 15/100
Processed 16/100
Processed 17/100
Processed 18/100
Processed 19/100
Processed 20/100
Processed 21/100
Processed 22/100
Processed 23/100
Processed 24/100
Processed 25/100
Processed 26/100
Processed 27/100
Processed 28/100
Processed 29/100
Processed 30/100
Processed 31/100
Processed 32/100
Processed 33/100
Processed 34/100
Processed 35/100
Processed 36/100
Processed 37/100
Processed 38/100
Processed 39/100
Processed 40/100
Processed 41/100
Processed 42/100
Processed 43/100
Processed 44/100
Processed 45/100
Processed 46/100
Processed 47/100
Processed 48/100
Processed 49/100
Processed 50/100
Processed 51/100
Processed 52/100
Processed 53/100
Processed 54/100
Processed 55/100
Processed 56/100
Processed 57/100
Processed 58/100
Processed 59/100
Proces

In [10]:
bleu_score = bleu.corpus_score(
    predictions,
    [references]
)

print("\nBLEU SCORE")
print("=" * 40)
print(bleu_score)


BLEU SCORE
BLEU = 23.34 41.9/24.0/18.7/16.8 (BP = 0.984 ratio = 0.985 hyp_len = 1527 ref_len = 1551)


In [11]:
results = []

for i, row in evaluation_data.head(10).iterrows():

    english = row["english"]
    actual_hindi = row["hindi"]

    predicted_hindi = translate_text(english)

    results.append({
        "English": english,
        "Actual Hindi": actual_hindi,
        "Predicted Hindi": predicted_hindi
    })

results_df = pd.DataFrame(results)

display(results_df)

,English,Actual Hindi,Predicted Hindi
0,The potential of our complementarities is trem...,हमारी अनुपूरकताओं की संभावनाएं बहुत हैं। व्याप...,व्यापार और निवेश में नए अवसर हैं ।
1,Progress made in a field of knowledge brought ...,ज्ञान के किसी क्षेत्र में नई खोज द्वारा लाई गई...,ज्ञान के क्षेत्र में उन्‍नति नए प्रकाश द्वारा ...
2,That was because their apostles used to bring ...,वह (बुरा परिणाम) तो इसलिए सामने आया कि उनके पा...,ये इस सबब से कि उनके पैग़म्बरान उनके पास वाज़े...
3,Oigodipsia is also known as hypodipsia.,तृष्णालोप को हाईपोडिस्पीया के नाम से भी जाना ज...,ऐरिपिपॉरिया का भी अनुमान है कि ऐपिडिया के नाम ...
4,Al - Kawthar,अल-कौथर,अल - कतहर
5,in tune with,के अनुकूल,के साथ गपशप में
6,"or lest it should say, 'If only God had guided...","या, कहने लगे कि ""यदि अल्लाह मुझे मार्ग दिखाता ...",या ये कहने लगे कि अगर ख़ुदा मेरी हिदायत करता त...
7,Reduction,घटौती,कम करना
8,"And when he came to it (the fire), he was call...","फिर जब वह वहाँ पहुँचा तो पुकारा गया, ""ऐ मूसा!","फिर जब वह वहाँ पहुँचा तो पुकारा गया, ""ऐ मूसा!"
9,"< small >% d smaller partitions are hidden, us...","छोटे विभाजन छिपे हुए हैं, अधिक नियंत्रण के लिए...","< छोटा >% d% d छोटा विभाजन छुपा है, इसका उपयोग..."


In [12]:
print("=" * 60)
print("       ENGLISH → HINDI TRANSLATION SYSTEM")
print("=" * 60)

while True:

    user_input = input(
        "\nEnter an English sentence (or type 'exit'): "
    )

    if user_input.lower().strip() == "exit":
        print("\nTranslation system closed.")
        break

    if not user_input.strip():
        print("Please enter a sentence.")
        continue

    hindi_translation = translate_text(user_input)

    print("\nEnglish:")
    print(user_input)

    print("\nHindi:")
    print(hindi_translation)

       ENGLISH → HINDI TRANSLATION SYSTEM

Enter an English sentence (or type 'exit'): How are you?

English:
How are you?

Hindi:
आप कैसे हैं?


KeyboardInterrupt: Interrupted by user

In [13]:
!pip install -q gradio

In [14]:
import gradio as gr

def translate_for_ui(text):

    if not text.strip():
        return "Please enter an English sentence."

    return translate_text(text)


interface = gr.Interface(
    fn=translate_for_ui,
    inputs=gr.Textbox(
        label="English Sentence",
        placeholder="Enter an English sentence..."
    ),
    outputs=gr.Textbox(
        label="Hindi Translation"
    ),
    title="English → Hindi Translation",
    description="Neural Machine Translation using a pre-trained Encoder-Decoder model",
    examples=[
        ["How are you?"],
        ["What is your name?"],
        ["I am going to school."],
        ["I love India."],
        ["Good morning."]
    ]
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7669c2c9e9a86075e8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
